In [2]:
import duckdb
import polars as pl
import httpx
import json
from pathlib import Path
import time

In [4]:
#Connect to DuckDB
con = duckdb.connect('/workspace/data/pipeline.duckdb')
print('Connected to DuckDB!')

Connected to DuckDB!


In [5]:
# Adding the bronze table into a polars dataframe for further processing
df = con.execute('SELECT id, batch_id, page, raw_json FROM raw.api_data').pl()

In [14]:
#Taking only the data column
df = df.with_columns([
    pl.col('raw_json')
      .map_elements(
          lambda x: [json.dumps(anime) for anime in json.loads(x)['data']], 
          return_dtype=pl.List(pl.String)
      )
      .alias('anime_list')
])

In [16]:
#Splitting each json with 25 animes into 25 rows with 1 anime each
df = df.explode('anime_list')

In [17]:
df = df.drop('raw_json').rename({
    'anime_list': 'anime_json',
    'id' : 'raw_id'
    })

In [23]:
records = [json.loads(x) for x in df['anime_json'].to_list()]
df_flattened = pl.json_normalize(records)

In [29]:
# Removing unnecessary columns
df_flattened = df_flattened[[
    'mal_id', 'title', 'title_english', 'type', 'source', 'episodes', 'status', 'duration', 'scored_by', 'rank', 'popularity',
    'members', 'favorites', 'year', 'producers', 'licensors', 'studios', 'genres', 'themes', 'demographics'
    ]]

In [ ]:
# Renaming columns
df_flattened = df_flattened.rename({
    'mal_id': 'anime_id',
    'episodes': 'no_of_episodes'
})

In [36]:
df_flattened = df_flattened.with_columns([
    pl.col('producers')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('producer_id'),
              pl.element().struct.field('name').alias('producer_name')
          )
      )
      .alias('producers'),
    
    pl.col('licensors')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('licensor_id'),
              pl.element().struct.field('name').alias('licensor_name')
          )
      )
      .alias('licensors'),

    pl.col('studios')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('studio_id'),
              pl.element().struct.field('name').alias('studio_name')
          )
      )
      .alias('studios'),

    pl.col('genres')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('genre_id'),
              pl.element().struct.field('name').alias('genre_name')
          )
      )
      .alias('genres'),

    pl.col('themes')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('theme_id'),
              pl.element().struct.field('name').alias('theme_name')
          )
      )
      .alias('themes'),

    pl.col('demographics')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('demographic_id'),
              pl.element().struct.field('name').alias('demographic_name')
          )
      )
      .alias('demographics')
])


In [46]:
concatted_df = pl.concat([df.drop('anime_json'), df_flattened], how='horizontal')

In [47]:
display(concatted_df.head(1))

raw_id,batch_id,page,anime_id,title,title_english,type,source,no_of_episodes,status,duration,scored_by,rank,popularity,members,favorites,year,producers,licensors,studios,genres,themes,demographics
i32,i32,i32,i64,str,str,str,str,i64,str,str,i64,i64,i64,i64,i64,i64,list[struct[2]],list[struct[2]],list[struct[2]],list[struct[2]],list[struct[2]],list[struct[2]]
1,1,1,16498,"""Shingeki no Kyojin""","""Attack on Titan""","""TV""","""Manga""",25,"""Finished Airing""","""24 min per ep""",3034074,120,1,4309509,187047,2013,"[{10,""Production I.G""}, {53,""Dentsu""}, … {1557,""Pony Canyon Enterprises""}]","[{102,""Funimation""}]","[{858,""Wit Studio""}]","[{1,""Action""}, {46,""Award Winning""}, … {41,""Suspense""}]","[{58,""Gore""}, {38,""Military""}, {76,""Survival""}]","[{27,""Shounen""}]"
